## Exercise 16.1 (coins)
In this exercise we would like to obtain some currency amount using the fewest coins available in the currency. We assume that the set of different coin values of the currency is represented by a tuple, e.g. coins = (1, 2, 5, 10, 20) could represent the Danish coins.

In this exercise you can assume that all amounts and coins have integer values, and that there is coin of value 1 in the currency - this guarantees that all values can be expressed using the available coin values. Furthermore you might assume that all amounts are ≤ 200.

Consider the above coins. The amount 33 can be achieved in many different ways, e.g. 10 + 10 + 5 + 5 + 1 + 1 + 1 and 20 + 10 + 2 + 1. The last sum is a solution using the fewest coins.

A simple greedy strategy to construct a list of coins for a given value, is to repeatedly take the coin with largest value until the remaining value is less that the value of the largest coin. Then repeatedly take the second largest valued coin as long as possible, etc. For some set of coins this strategy is guaranteed to always find an optimal solution, like for the above tuple coins. But in general this strategy will not always find the optimal strategy, e.g. for the value 14 and possible coins (1, 7, 10), the greedy strategy will find the solution 10 + 1 + 1 + 1 + 1, whereas an optimal solution using the fewest coins is 7 + 7.

Implement a function change_greedy(value, coins) that implements the greedy change strategy, and returns a list of integers, each integer in coins, and with sum equal to value.

Example. change_greedy(35, (1, 7, 10)) should return [10, 10, 10, 1, 1, 1, 1, 1].

Implement a recursive function number_of_coins(value, coins) that returns the number of coins in an optimal solution. The function can implement the following recursive expression:

If value = 0: number_of_coins(value, coins) = 0
If value > 0: number_of_coins(value, coins) = 1 + mincoin in coins where coin ≤ value number_of_coins(value - coin, coins)
To speed up your computation use memoization, e.g. using the decorator @memoized (without memoization, the second example below will run for a very very long time).

Example. number_of_coins(35, (1, 7, 10)) should return 5, and number_of_coins(100, (1, 2)) should return 50.

Implement a recursive function change(value, coins) that returns an optimal solution, i.e. a list of integers, each integer in coins and where the sum equals value. One approach could be to modify your function number_of_coins appropriately to return a list of coins instead of a number.

Example. change(35, (1, 7, 10)) could return the list [1, 7, 7, 10, 10].

The depth of the recursion when running the above recursive functions depends on the parameter value, and it is very likely you will get a RecursionError: maximum recursion depth exceeded when trying to compute number_of_coins(1000, (1,)). In the final question you should convert your change function into an iterative solution that systematically fills out the 'memoization table'.

Implement a function change_iterative(value, coins) that returns an optimal solution. The function should fill out a table of solutions for increasing values of value.

Example. change_iterative(12345, [1, 2, 5, 10, 20]) could return [20, 20, ..., 20, 20, 5].

In [1]:

def memoize(f):
    answers = {}
    def wrapper(*args, **kwargs):
        if args not in answers:
            answers[args] = f(*args)
        return answers[args]
    wrapper.__name__ = f.__name__ + '_memoize'
    return wrapper

import time
def time_it(f):
    depth = 0
    def wrapper(*args):
        nonlocal depth
        if depth == 0:
            start = time.time()

        depth += 1
        res = f(*args)
        depth -= 1

        if depth == 0:
            end = time.time()
            print(f"{f.__name__} took {(end-start):.1f} seconds")

        return res
    return wrapper

test = (62, (1, 7, 10))

@time_it
def change_greedy(value, coins): 
    coins = sorted(coins)[::-1]
    result = []
    while value != 0: 
        if value >= coins[0]: 
            value -= coins[0]
            result.append(coins[0])
        else: 
            coins.pop(0)
    return result

print(change_greedy(*test))

@time_it
@memoize
def number_of_coins(value, coins): 
    coins = tuple(coins)
    if value == 0: 
        return 0 
    
    return 1 + min([number_of_coins(value - coin, coins) for coin in coins if coin <= value])

print(number_of_coins(*test))

@time_it
@memoize
def change(value, coins): 
    coins = tuple(coins)
    if value == 0: 
        return []
    
    return min([[c] + change(value - c, coins) for c in coins if c <= value], key=len)

print(change(*test))

@time_it
def change_iterative(value, coins):
    result = [[]]

    for v in range(1, value+1):
        result.append(min([result[v-c] + [c] for c in coins if c <= v], key=len))

    return result[value]

print(change_iterative(*test))


change_greedy took 0.0 seconds
[10, 10, 10, 10, 10, 10, 1, 1]
number_of_coins_memoize took 0.0 seconds
8
change_memoize took 0.0 seconds
[1, 1, 10, 10, 10, 10, 10, 10]
change_iterative took 0.0 seconds
[10, 10, 10, 10, 10, 10, 1, 1]
